# 50 Mini Crosswords: Your Heuristic vs ToT-style DFS
Self-contained Kaggle notebook. Enable **GPU** and **Internet**, then run the cells in order. The benchmark is disabled until you set `RUN_BENCHMARK=True`.

**Exactly 50 puzzles**, sampled once from the [original ToT mini-crossword dataset](https://github.com/princeton-nlp/tree-of-thought-llm/blob/master/src/tot/data/crosswords/mini0505.json). This is a custom seeded random split, not the paper's published evaluation subset. Selected IDs and source revision are saved.

Two methods: **ToT-style DFS** and **five-factor heuristic best-first**. The [official ToT crossword implementation](https://github.com/princeton-nlp/tree-of-thought-llm/blob/master/scripts/crosswords/search_crosswords-dfs.ipynb) uses DFS. Our baseline adapts it to a local model and a shared engine; it is not an exact reproduction or a claim of current SOTA.

Both methods receive the same model, prompts, revision rules and limits. They run one puzzle at a time, printing both results before proceeding. Reference solutions never enter model prompts, pruning, stopping or final-state selection.

**Not run on a Kaggle GPU during creation.** Offline tests use mocks. Other benchmarks and previous project files are unchanged.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','-q',
    'transformers>=4.51,<5','accelerate>=1.6,<2','bitsandbytes>=0.45,<1',
    'huggingface_hub<1','pandas','matplotlib'])
# If a restart is requested by Kaggle, restart the kernel and continue below.


In [ ]:
from pathlib import Path
import base64, hashlib, io, json, os, zipfile
PROJECT=Path('/kaggle/working/crosswords50')
PROJECT.mkdir(parents=True,exist_ok=True)
payload=base64.b64decode("UEsDBBQAAAAIACF6YV3a0AL0bBoAAIBMAAASAAAAY3Jvc3N3b3JkX2JlbmNoLnB5pTxrc+M2kt/9KzhOpSjaNC05US4rB1M3m3EuucomczOTu7pVVCqahCxGFMnhw4/o9N+vHwAIPjwzu6uqGYkg0Gh0N/qFhk9PT98/5BdFnibRk7NPsuQiKvOqesjL2LmVWbTdh+UucN7JsIy2TibvZemUMpLJvazgx0aW0Ek6YVY9yLIKTk9PTzZlvnfisA6jNKwq6Jbsi7ys2ybfCas4iWruWYT1Nk1uda838HiifkfVvf65DSvspR//qPJM/95bI8owi/O9eZL6V53sze+mTAFSUMoPjazqk5O/3bz/8dfX7xzhTNw6r9fxpnJ9x93KpkyqOonWt9Btk5RV7Xonr29+ePXbz++hM65gsg8f13W+k1klvprCx3ewJQrTtBJfqyf5WAB5khz6XE39E6f7wR6xLOotvnRuYQHRdr0JozovxVcMAMiU1YksxTfD4Xcyk2VYA3SDxxzgyPswbTrN8+lwcNbs11H9KL6d/eXKd2q5LxBWU0oxDb7xnaJsMgmrT/MHaJjOB+MfZHK3rStBpPjQhGlSP4ngq7nvVE0UyaqCB0CGl4cAnCivahHM4FcMElRWNGA297yTk1huoPEOaD3Jb//wFjRZKQGdTHM/qLbh1fybCXI/iJt9UWFXmA3Yut7Jp0q8LxvpBSCSeSwnnhds5aOCqWZ4KJNarhHABAXPd5BQUs2GLcBZFEF6611TU1CEIOV1sN/FSTnhB54KCP0IMrLOdzwzQan3BQChgQ9JvV1XzWaTPBLAgH+fuwF0ck33gLGq5WNtr41Q850ki2FCceXBbLiwJLsTblNvLr51AUEcXsoiDSPJOPM6N7KOthMQdbU0xKQn+QE85oXMJr3mt/yNzb6zlWEMfBIH97dKlhevQN5qd+EaJXFRyop0g3sE/HCb5U0NgurBHgfuVQWIvVwYwVH81C9gzjDWrClgGUDbNeqJStaTMs9r4K1MZUSCXEkZwya5+kYtCXTNmyRzmqKqAcwegN4nuM2unSrcF6l05lOnyRJYjHNXJnEF+myTl6irQNOBfKROFeUlUJO0FmEHM2r+428gL359LuP3YZZsgHJrJUc49tKt8qaM5Fq/DJDBLvVPNt0hAcGsJl5Lr72sQ4BEQpHmYVxNuiOQfiw3nmcGIVwYt3S7tHNXzgvRo+eis6XLMKmk898odjdlmZcT953u7WBvJ9qG2R2MchroFzob4OPWUfyCzQuGAdTWU+C2uJThAypWJgXal+l8OmcaeIz97VMtYc1mBNqMVuHrLQ9wOrvZEUIvkunL/dwVwZFpV+iKHJBwC2B2JOs8u8jS4hKERl7km4t6mzegxS7SdO+aIVG+3yd1l/K8pTbutq6LanF5GRZJcAfbqrkNoPslzlJdHvDreMnjq8t9WNWydD0P8NyGCjvFoyxHCxVsGpg5RNClu5xe/CW82KwOX0+Prq+w8BYjjPkpA+2QxEPpt2gPGxhW0CIMNFQIA/vKCIwKajXEXWF94AmPl1UZXYIpvETWXpq9DsvpMLDH5FblmBfgVYDtkcxj0+Wfwejnn76/+eXdjbW6Z2SKVSkLFQpNv/+7X397+/3NWsEL6se6N6iDs9ffiWTqiNUJirrAn5pRgr98JLwg7dmRTfEJkR7ouu6jpwxihJzoSqZZp1J3wjkcwXQ1f/6ZEtmXLHeg/ciYPPrOJAJRAkcslTXIZ+VBOxiXZo/2X07UJJYeus1DcAhhD7nBH3mSTfS4oCnAZ5h0dE8KJoXAey/EbOrApNhCEKDlao4tA9l3l68u/r46XM1R7rnvJ3TTxv1bmMKS9qCXeKnOgVZ3tLcAESQA9QQWJmzSmvHwmY9JLDaWKePhi+nXMSKheEdtQpGNliUU7ao8bZA7gtHlSSNwQBPYNkT3NEEzyiiQMa+0mtZkMr297+bTsX3+C6rdsIbeoEnQosXok2ZRrfDTXFZrbnnOnnDwlr4mPVEK2EJa8/sAW1kxEHRgK7ZOFA2ivAEHBBHWkuEryq7JsoreWnBD4Ij50OVUMi7jNY5bFks3QcOEolmgEKoFrHxtVvTmUbtEvfeeA4z8cHGNQDYmQd8NuHaipqqhvSpS3KwoivUWCXkL/NrCuPf5e3Bgb2Fy1/u4oQZfIv4844ymFylrWaUBr18rQ1qU+b3MQoyrlMFVeFjOa2cmn0AP+rDCa9X3fMpK0tdU5hFf6OguyYqmdsKyTjD+AA8HCBQldfoEP0HmY5ArjvMc9LWD8enId8Rgp51tedgt7onFO/+emBzAoD3SDyi7ewEMU1vJPQ4lQes9chxVIzuMkYQwa1Kl4Kh1ogXadiUSjl6eYTxC3+cQaOCU+PTdnLyEfueLuX81901Agk3rLNzLkWkm7tbtQ3PvXe8cLDJ1/3J+PlNwSEWs883kwVKsWqku3bW7OruaGyWNg32HYvCOXn7oaWXlRGDzwkHrmWSNNG8tdc/qGoH9mRQTi248S0/VJgrfJY1e0RwJrnft+gxp1CX5Ps82IDCAxB06ccldtkd/2dLGZtEaslCY2WTVJoa1qgoNKLyh5fsWddpA4K3yfkBDgyxXSAyDC22AHlI+zhaC64DNKWCQQbB8L1X8LSsTEigSA/CsqnFLEt38JKs9bcem3wkSAbZ1z1k2NGw9WqsF/5JnUs9FS1wiuBV6uMzZfj/wusGSCZJcFolr1cYjhcuuWR2Wd7Lm6LzPd0LFMxKXgwYsfSdP474ngGC7IoedUO9BHDUhPuIQnova+adi8QuhZK+VxjFhpPk9H0B7fbvPC6MOZmWd1eJSbAGqGzRtCvOTk383eacT+t95V8PKVFSM1FvwCMr/uL53NpvSO8paLABNDAO46cPC2YCip4aAmwq7iXcwyl4SY1qs210pggaIEOvYXnklmF5R667Lp5YAj4IAcG+vLzj78HEyDXzwgCezwH9k3YbZsCCpNiDbwL9Hj/WSmoeDo8cIluZM2r3rv38q+JdnpE2PIKwh8I3BzVOOz0CHiZ5662zn3zO1oTfuoVWniXdcOAcCuExWR+f/MGGC+1AcTk8tBbDcsXOwQ3lhWUk8D/tr8yQOvGmSFW6/0waIn6YyPj26LHM4kPX7bKrVegQGtgxR5aMsdNV61ewnEdikNY+PcLxZH/UPeJXeJahsTrYkWZiuYZJdB+AXzk+YuUmiMNWG0zgpBPsWpBpN9zZHt2SQTQVTLMF1Ce/DJA1vUxl0rE9vDWeM2ge/vzhQmPSm8J0L/kWSbfJElPCruLPv6FwjgNnc+Q7acpAocNcwp6lyiWp5tCXFsmKThWTSg8kqgs4kZUiAOQBpc34osxOkdPhC3BKA0L/VKsEisl8ZYoN6NRMpdWFtHy1ze1bqAmGTbaRBDzjI5p3iz5syf3yiHpsGyYCivWOXsGqyHScEMWeJohWCcw/hG+rFGCM+0BmakzIC7zxQu7+qhdqSEHNqhAB97+zqbJSel0Dqpdumkt2uz9PJrhou6wyrZu7AH+asq8LEYnw7Gz3CZGeTmZKMYsSvpgXhf3bO1vw6I31TfShrxbYPZxqUSeslOTiJT/+EhG1ETzzNWBw6PvL6QdACVW5apVwUKR+WriIkOFwb6+Ec3iiK8hvzcAFvFKGwXf3EVqQJN/IvhGHoonq3jyfK+vy1icE6vqvzYnJDihhECLRugfZJdfnxh+/TBBSH8W5uHsEfv6AFYpYUPBtY2jXGCfuivsgz8ND/892vv1yD4GpFg7nUpESx3SYxaCGHziSMW4OMWa/RRqzXGBxufMqI+jRH8idQuM4hJLBsMXYKqJOg/6+pwfQX5pd+AcMF/X9iZlR5eZqPke+kOqsKtBYEhZxjyVMJgcFTBZGC66vUkHDf5Sm4afPHudPGNYHzltlLlMBYTqWyYZ8iXZz89g/Quj4lwfMGT4zKXZw/ZIHr+dZkmINqp1IItik7jOREd9VBWBTp0xqitHqNxycp+kx6IYaY4ocQbLAfxvHaOq5h+Fb62LZBnVkmOLPPr2CarMrLSrhF7RLIqpBREmr557m8luYUiXdJriZJVLs+LlGvISLHOBDjc3eFuapCLmcrC6JaQ4ePPobcfppg+ss6R7LYywLRykbrUeJslRjBBDpOWrED7XWfRNK7zgRizsOeQ7YzawDBMmx0Sn1MKP9xze1RE4f2yzXskUmba8PP7gG8WXXChTozkw+a0rzaOF9zMkVY63459QuM/bEj4NaXGvvdUOHiR+bVs6Ptd53EWweBBWOuMzn24Z71G2S0WO/ElL4LMQv6pCMSVU8QFIHqhekn3jWo4rIWeNQTAJgNZ4c6SUA6bOLxSaacmvWejuQWsP+AYcLiq5GnszNmqn92xsh/BjIyDYtKxmPoXBCiBoSeJhaMwnLqZ4vVNaahzCugMkVV7cRA7I/Rv1WPQMBCoiecpxOAyakh+LG8mK0oeoZhyCQrkoQWH6dTPvoSnlcDVUCSN6Z6Ykl7xeDuV7uk6GsD0i4+7xHVlI1LHBPFyLbMaBWwr0EfAokUnf26bLKIqKi7vOSNQOtlf48o4Wlr9g7UIei7xTN2JyJTRwadDvw+wJagLdgzPdxP8BfbGBiDtp4fcLCwTu5M49jp3cjhnZkIEBDwjwGoPLyMe8jQAf/UsoHmqUNr1dal7VSDZspOA34GBCu53oCvjw6PAW7oPW01cFjtOsp3B4E1K1+Lapmw6BawGVCK1fT5AiwnGD2wqJivsJ1aWGD+QClIdWpqNvK1qgLBKAOkDbheJSALdfoU2MqopdNLobnFbic1uhStWfQ7z85pBS+7nbVHPHYYYLlSLm/IvGRf8JbeuB3tOISvyh8A+BAeeQGP9ZrGuLaIACvQ/qh89FLLjG/kxebKylsuvl35s28850tncnV29tXM3uHkpbBxUe6ANVYQW40EMoPFwMZa1LVaXUuTYBQjzH64xDjcMGcx/So+9o7zgFjPHEcz1ngEIKys9/Mn0QoaD1m6asHu6oVQPxfDNOJbLGbAWqSK8mfX6qwZDK/SUe1Bcy+1qKsKRDsfN1jnrt2TYaIOiDBGjIOCDTdQr9zBgtSLZyjUSuhb2HRgmNTKXmWODMuUAmS9m5yHECNaMFhlU6DDirEo6AAZXztJXXHwCZE0RKMZeqyYvSprc/auKFKXoPODHp4aSau4xP3JzEvGEWeBmKEK7/G4RFHLeZJ14PqDWpNxWttKxljyoVP4SSHFT784Rx228i5R377FZf7hjS+7ydIk23WPJvUQ7TZq7fJCZEj49nVHY7url7SIEWH9axjtUAWGESlYTCdrwXW9EZtxLma20TgX2fmzk3bMxLlo+6kmS6oHlgcAj5geG0hvsi4oY3XO2aFph5k3Nu86OUv8UL7SVhEWAJDDPt+HSXZOjyL/vRGq/5Kju6HCuoFgkttEANpd/9GUZ2f8iC0Gpo1NcTiqMyIMSVVulJN5IO3k86BXo8CPnotzb32+ouoQR9ariz1qMPztOLVwUx24pMT2xEPBkZk46BfL6YqzXsfrJIua/S36UdbLaypQjNHFuK7LMJJiuboGdV7hUarut4ZdWz+xjXjYgn5qk33o+GkQ3xnD3VY92sZb52yEopDaB5eYyp5psvFW4ZM6u4Xd5NnVdGrvaSaGEKZ2c5GBX9wuEPzvj2h/6ouzm9TSTj6JNNzfxqFTLdrk1Xjuyetscj7Y1jmrNnmFkzwzfjhcTynM3J8zXL8GO7wHz4zGdN0faOAE4EvRzwAuBmeHmp+26CsPZbJx35R5kQMrmgICM+eA4E47launq2NbqIDjEjqAy2Dv87EmHejRSRxlX+stmDg8nQ8cd2BK3Zsw2uq8FvSTmA1Ln5wNHtjdZHd4aq8LWQLnf/PG2YP/QPVQmErnSj066CMQ12NTfOq8EEJa8I9vJZ8vQqDovKF0vUOqHjuTeW7RGJlDJakOp8ig6nSxPJziecjp4nQ7O/VPkS7w+9WbNz/fwGN7inS6mAb/dgSKBr9n7nnnKIaYqhLkz2vjgtkVm02H8YMy0a622a5PUjGoJ+7ralhtGgMSYrnqtNOihJ4KXAHwNqjN9ZefVPfUkWPhBQPqQUchwReoJ6nDckHodqTOXa2Gjtj4ZMq0DORef/CMSuA5PnbmxSCvXN91PQ9s2gMmGK6R8r1e2KR6qdKoADokxcR7DrPeYfH2frWcXcxXro84fARFxIciEeyHSa+L2flkTlCxYboClXjvss6cDmcnqRF8sN6KkU+n23Q4PIYv9cE9iGfRaIlUA5WHZh/BFd4GYRzr42otRJg/BXmesOViDFpFdT7z210g9MlpS+v2JZB7GHUgcfVEQ7kgS6fnJ0fTWCyhtZ+vlbLgCwusVX2zBaYeLGVszVpVujdchy/B8QcdZkqkSNc54V2IQum8+vlnp8YUPR+vgtJMG3Rh8UoEONZxq5FUunuoXPDj/hdyZi9DhnShNCawuKmS2wQPOkx1SeVsw3sufkhZkX5EcxHwN6R69ZELnhsaoPnGZA6yu0sIvTA8AwVdgR6pNk84DSblH7Z5quvVnp3mN5BWLMuWJFXLqT9bBQ4ETnh3A2hwm8r2kI+W6VTbvEljUMQ76bxx7mX55MD+fHYCo4VRy2V3rIcT0LxT//QDqVr/tMDvOahc30E5554QWJQtB0kXj8/we2YdeX9vWH5Ijr9nh476jpTu1kfVftStvtCC1hdudbvjGY2u3yqNPrgKYmt05Ri3mQ8s24fGfshPziBqOnjnXd9Cw65NYzAhhcGKd6dq7qr/oTZW3ZTy17As9X/7hDlwcLZ1A5Kq5FN96jwoXrKhs6LnuhTw+CclI5eA9+yhRGN5B750P7gEkh8KfhhqDULGLictuYDRL60C9ghpCsyxFqHYi/x8hsXDVei4gOY0uhWTdrAfmsyQm1w7lEzsD95Jhk5Pkt2D3wIvt8ndlndUMLTjwQejVmmWZbJaAiVWSkMHxcjrYiR6swogaKD30moxQYe3aOMP6tYB84XzA+YgqdSiUpvcsQtUOe0ITlmzpwN4UjNAHSe0SidyUOtpb6mAoFqOcn2ti03g/Gp+aWPAK7DcqX/aUJhIYBAb+O3tMuKNGLj/frt0sVRQI68tSdGCsxpP5/dEEWNP/QAGjNLXndjKV8npblw2uFTSi7R6DPwRZA1MxEVrmBlNqtWBVYLZwGt0187rH97hQTalniuu5jCld1wZE0Y7JPwOMzldT1AHOhBLIEfw7hfIp15cN3yrghaVng41YXMbKJqwSDeBcHwsduyjYgjcVdb/QMjbzUSQGRSHBBbpfToSHZbhHMcXrCg2HvDypEuaE7QabrlK8jGJcrm7BBq/b8O5A7NPdD4ehUhvGSsKtVMFqox2LO1gH4iB8OC3qb7H2iyjXJRVNZtyTcUjemu26mh0HwzdXvtAR/e1G/3u8Y7u0mkdgoU91B7TdUXDbw9N9SGR7qCe/d5hkX7dbR6ZVGfv1t2lm3Zfs4dpVlkaDk+IFWf5y/NJNaokGMqNulyhq7m7dXVEfPYy17BTGhj7RGVaoRiWgPUAYandcDXI5h6oNDVlg0Lo4SM1hKDZ2v3SVgdezsZuLGB+T9U4tkA9XRmJd4r5zFEtt3cQWeYPoMHt45LuxQBT8K72paqjVzctXC+4S/PbiXumLjipghQdPAJwXc+h3A06fDdAsAi2Wqc5qNGougdodOPTfYDwUj6kSSaFO5LqRy9w027oTSJTvK2xa8mIE0OMaQr4O1vNXV1TKr8UMGfwGhj/P/Q42fgMCsQMZKcK1X2N5C4D6bESuTyaTy74BipG21YjTj/B/9StGw4/pDiUS3MDYW1ulpQaY0QXZQRLlx97PYUYjEVfELqpdCv02BO4Rw2Ombe3krce61yQxX1YJm0eY9Nme61Mb5v8o9smYlkO0S078zMIxGuALMq2IkNrtACKql1h263y0xk5BDyrZReJu/KJi/1Z7F3f7WwzeO7tYWjhE13ftXUaPA7UGLT19Au0dHWW2z9tzB+QAGG2ds8BtRVt83KJP1taqYVcWoui5DH9ZItiiuc77NFuHcziDTYP93r6F/bNc5uAakwMErCNvK50Py/yZpC67hUmZRtoID0+JLGlS7RAWETFIbRNtFQtxoWutzcA7rEDo/WXVX2q6Y59/bOzw8Y97I/rw+7oLnDAcr/SitjeMa1KmXyGpBmhHBU579her2McLV5YnIVIfq0R/hfY+0kWMw7/CH95xDM3ubRA8rWqnii01evcqsxTk+krc51KGz6XVp4L1xh8fTUwYPwaTdhnl9Gocy59weyFmE9HDrTebymPD3xIML1OlRAJ5nx0bn8+7V1sVHAPz10YPD431esGrwlgAkZdEv3ptQUT/85At1wY4GCj+jsf9gt9u0fdsIjSvJJUw94F0N709Gdjh5f6ME8P6P7tANbWFLVQcZNmldAsa+tB/tELkmgm4+cuIxO/1+tNksr1unszv3tJWV1XENrgdbG33BzrD1no157WWvVWqNvbo38V4aN3LAfVJi+EhjJC7ZtWyDp/vOAj5ST9KgSDPb1N82inkAeWBPjotqYDH5UueWR9AZPGqVzwF+/0CV0QxBo39RdA7AOb9p6UktdOukhzt3/Ag+cAG/f37M1vf//7zzfqIvT57HgJG+nAg5aY5VwdXX+TNtW2V/mGH1C+oMm0WsYMFAL58kqIKdtQ9Wq5WNhnqhrp1r0hQMPcmcV27dxiLVIHu/WBwfRLkvRnUJqk7sJ9djUSfoaRvP6oeEiomsVJrzjxUpmdS4UzqaJL5VPZbX63iNj+MMIcOgkuHdAjKSkMg0eKCEZBfeH8R4npOQzrQPY4m1nmcRNhHRHWwr/64f3NW4f/QIuyEBWXFW1BeWV85PoRNM09cIru1K07tw2+AVmNvLnCuxqh+BCi7S90KNd1VFttZ27bj0w4Ot/AfBLJPy1zzJpPwiTVoC8Z9vvpDek4Cvbi6qo6qjO3g6LjacfdOV0tgumXR30O0/bqOUG6n4pKTTd+xoN3Du3NC3rEdpVrMC9s1+mjioG518a57Gxigje1FBepPlV0tU/ovKrrGpi//WNB+n9QSwMEFAAAAAgAIXphXfE2/Xt+BwAADxcAABcAAAB0ZXN0X2Nyb3Nzd29yZF9iZW5jaC5wecVYa3PbNhb97pn9D5zZ2QHpoLTkrZJYKTvjJE6aXddxbbfdrUbDgUjQQkwRNAA6dj36770XfIiiGFnZaWf1QQ8I933vwQHFIpfKOJ+0zPYSJRdOzsw8FTNHlH+cw8+96rvhizwRKa9/F5kwhmtTCta//IWMbmpx0BY18pGSWn+WKg5nPIvmDtPObG8vSpnWzjt2w9+kgmdmvOfAK+aJE4YCdIahq3maeGN89xW/LcCIDibTZl8ki8zYTTQHT3LjjRU3hcqcw0Gz6ZpnXDHD2/uo5jymqVgIQzE43FAo7pUu4GvNps/ynGexWxlpNonEKZd8bZgy+rMwc5ecK5lLzUlLG77uWFrw4JEs5B3XZDx5JDqVhozJfEgowezA9+Pz89MT+BnJLBExJIuTsX+0pM5q9+Fq9/nJ8cVld/fL5XTZGOap5uPaMgQpsuvStgD5ASW3KEBJDh8vlrRcHpbLL8rlNXVVdmMRGdfwexNg+/hxsci1a614VGR5YUIjb3img8MBlYVpLQwHkHpwN9aBP6RGFVkEpYmDdwwc9eqeuMKku01f4c83TNflwaLieqj4ndBCZiHL4hAaznCV6apnGpcxVTqY+XnKIu66hFBvH7yA2Mtce+slB/NcmZPbgqXuzJ9JBk0rE9dq8Sbj0XRTbt2C/UVHlLy+ODl+u0293TkZgMZtuwjxP0mR9Tgjpk4ilSMckTkzP+Jpqt2R5221/EGfyYy7685CLk4/nr0/uSDe7kKrCKFsa2XRkVS8WwYotZkH5Pj1m7cn797/8OFf/z798ezj+U8Xl1c///Lrf/5L+gxfqQLNlgqtBmrfvQnRMr3jMZluLV8pSELyzIpNhuNpo8DCEYuiQrHogUyp/3I3VfuHo0ZHyrHn2loG3WTMiviaY6tC7h5so2rOVDTfaFOAjgZnoeMROJl6eCsUj4xUD24sVICY7IYhbglDD+DJZsH1/JwpAFAPgdUs8nXciZLr4HF/f+ZD4o9/Pr2iZMHuQ36fswyHB+Dgn8s1AWyqBTdzGZed9ePJ1Q8f316ua7WaLWwHKwR3vVdKSlP6CY54B6WeDUnNNdqGobksv7mlLgrOUlSBEES/PfQ2JCHmIjU0BNEy+AmJAHfIFIe6tEYr7ahsU0Grtu8VB/BRbqkTFDEYobKIW6ROQXvZFbWckYalFcKBOJid2BzXK31RYDs8Hf6rGYdq8CDl9ZbmTPpSag6356a0/GRq2gHSSvEOAn1+0jIGr68pMd52qsaHo/Ve7Guxrc2zaj1Sjh7pbSU7bi33L5gAtTDnr63QpZE58g6rHrbcuGQO+CpBWU0nCB12+mQjG51M0Mm0iw4LlkJuFjwO8SDtYkJ5GL5msbuKv0MqNgnOPmtI0OqYJpksqR554nwebT2e17L3l4DVZm0xfOihBr1ApLeiG9l/unr0cbmjDnBZ8zBhIgWWqOlwR7EyqfRw1K17zAzwGRPmIsuAktlzIZPwoT/DeWJL1G0GSK3lGRX9xVdz+iuWXXP3+aDTHJYwBKnQxiXHeHABnuAScI4gmiv3+eiZODg4fF4vD1fL/4DVNV2V+ZoJT9rQYqVbKIdhJhz4v1uotOMTUGZyEMkFMG99QNB52FM3bItNAt+dM6C4jOx/O1h6PpBbGUP3dJWBNPwXV9T79MObk7NLIGe1xhlJRQQ14KQTTddeFV+PoT+z2am9Fvly9glk3RklNk2EamDvIU8SWA3sUidrefH77ym3JBMAHFTWHWShbnszIiZX8h4dPYVZuPsxn+BFoKSXOdaokl/uoKDaSvtd/aLwmTQfMhdpXWEE4pStTSpZrF23BeoWSpG2jAa+BTRMNotDhDnX86C1dwL6X/CycqKUVN6411N6dNQdWkBDAI8QT8MFr+4c0AQxkju45W7ePP4ylKzbYWIRXsRBQm4fxZJQnElYbs1mndCA/GYRoAsaUM9p38m8hS4Onzyi1/7/u3N8J0XsDAcDJxUZ145MHHxU8A0k1CmPn1dOJmEoWQqOJVzhTdaRyKq1v1lNO0MuHO8ihfushuqJzHRv27a9isUCEg5TA5aaxmyRhccl3ZgfGxI+V9iNdH2V7k2GgIZon6EvCto7ERBVV09IRqZBMBrYmmqsaRXxDtJlOzjIC3C+V6O+syMrvLAnTHtIKzCF2bxO5cwl+9WowuUUumC7c209OYxbPfB+pO9AIb8X+GzA+x+mvKdQfRT06Oho2VRvHQHmvFBgXkRhwu0Do42ZR4i+NEjGyqcCbv1Ex3tWPno4ojHP4SY8pLeB/5Lmgf9iFUoC0ivNdIIoWmrzpi0KBDlsJbGHfRg38Sh+NDJw3+Xiem7wNtIr2jTG4Lvg7rtgaBviDhsi8e2THcz5lyURQ4VUwjxYvze8/b61oYmpb2M35TDaXKkiBxYa4v0M3jJo2nCOJ2Yk9QZPqkizwmp+NW/GtnEuYCbFouwbgBkWgabYqcgf+T+x4DKeXYhw/yxc8Gt+77ZDo5uhff2Np+tn+xT4U33FUjtCO0V2k8nP2W6+7gl8lpwxOK7DICAh3LdEFoZkvHpkDQtwXv1t7w9QSwMEFAAAAAgA4XlhXVDJ/kuICQAAYBMAAAkAAABSRUFETUUubWSdWFGPEzkSfs+vKImH29tNJ2Fg9qRB+wAMsLMrYIBhT7rTaeLudhJv3HZju5MJv/6+KjudwOpeTkKM0mm7qr766quqPKK3xhlqgo9x70Mbr+hyQf3w9avVcUpp7ylqFZoN9d6axug4mXzurVctLV+Ohy4X97/qIZiYTHO/i/d3/u7+d7VeWz0z/cHVS0qeygO6SaS7WreRlLXkdKNjVOFA0Q+h0aRcS361ssZpSjqmOKNXTtVW05vbz/LtjUs6OJ2mFAYH99LQU6OtZX83Wp7Q8uPnd/cvXr17+evb5x9//+UuDHpJxvELcorfn9E7T++tVZ0iH6hXpqXntzdkIrzSrW5ndK1XarCJOt9qe0Uf9trN+b+L2WX1jxfVjYspDA08eVrVJtG710/JO/wTZ2d0B2vOJ117v6WNivyBag0f9YNuhqRbfj0DQ/WBTAIoQ9r4YNya1Fq7NJtMXj2oJtkDJ2Zw5sugaR0Mwxc0YrW64Xv2BseGREH3VjW6w1FaBd9JxLhvbZyyhMTQskPGF5eLy9mf0bvllIbI1vJNxjN+uO9icfFzDmDoEaRWHTW+6wxibVVSHM1mejJ/cx0lNyWJoIp2UWcX1Y6hvNsAV/xT1AwxwbGA9/En9pYvZWTg3d8iyFdbEze4dGSlEIHiUCO1kjW9U3ZQ4m3mKt88RJxZIZMJMLn1jJ67uNeBtvqQwepV5FfARQYlNhnm1eBy3N4BZLUCuY6cD+BWcPGZvC8FcMCzRpudjvl1/iIhn40dNJh6DWiYfmqnjFW1QWgHar3OmUcQSmIjhUgccJIMCbmoh7GgjHg+mTx6RG81MtoWWDfwH4B4l4K3qMGKlsmn+3YVl1eMWxXTASi0uk+bamUCwyUxTPnQyrRsrgKUWu7ZGNsG7abF9kqraIq3fRD0xGytmi2carb8IG2CH9YCCtzELcpyHSILOw79lF/rG2WrfLNqVZ8kUTnFiqkPOjNPg2+HgvzqL7nHt12f4vw808En33g74+g3o97UgFUiBhJRdZoauG5AUp2xO90AgeAyIT4xgnRkfZ2TuUI01QoegkejDS5CBT4gf0nTxlukJUkV9yyKiYSkYJdZO648wPFR74zUlcpfNhbpiJIMHEn8jdBbXjk/+MLDw67kHpSGV0BVAgtyJ3wNg9Wow4Gtw6Mq+7UynJApspbg6yns6QkRgqToUJ4WUp0nHznWkeMT5FasGRVUpWdhUW4rVfV6gGxnBcK9wMTgc1280+0z9sNmwmSlEkHl3CP6gPJJDooPBQ7JrMT/rDNiLR/o1IPpzFckcNn4HRxea/qRPixJdR4Xl9CYyGOqS2qPr0/pNhdOhyaDgpbCANCUjK6hZlsdgPUbMBovQcPxpdM4itLwPsIwO783Tkohw8vCAh1d4yQhlyh+yFjf53o9NgoLvyHiPW7KORQvskZd0ZPFdLFYQIASIjUOsP5U0E1+q5mfTxeISVrZxYLppVzMxM0BXCyQXmQCBQy/nuQsBaiCgcGfF2iVUlxyG/BDOAgQ/iCrOux0ZjjnD1reWy3Ehzm/V6xGtYZ6agQH+rAXUNAcEJCuh3YNYevUgbTL7QbdaMh6yr4X5p4IVhSEufgnDgvRcXsD1q25Idw4pBHJ/u3T+3dcF3HogLsRbxV7GJNcIe2YUxONRYlYlmD0alA36+Q4eUwmy1/pF5o9uaQP9BP+LkCCimaLS7rmv48v6SU/x98/liyhH66y/lVByKQK09BAh7EiIE0FKw3LTG2mHdSeovUJiTEOn1qOTkYhMnjZMGCHscAjK9btt8bOqw42IiCLq8OxbPaoQX3qfyd1LXUE4Op8D0SxLo2GrVxf4ZUA0iP7iChLm3AHWg+CmtG55Q+Pq9u/L5+NqhIzWbhI8jlx++UVty3TZacH9ENdwaUtG344XIEDWb10WzEgYGMxijNcRhggC66VZLSwuzU79CRxJFeDkKiwDIJlTd/nVv3vxfTxf2RqHMOXPintEBcyZRsVMRc8z3ZiSYkuPURIdMb3QlfE9gdSgobbDd03oI1qDC9R+mOfVrC+AQ/HBjhWngjEXwCOX0L64cOPAHlGn5BZ2If58RAggJRw/hEYatWDzv/UZr1JeVppVAh8FSsaJwXR5EQBrX1+7xn8PpuS8lOQNEl/MaJDwKfjcnutFScv340pZXvWm+P0SCyrTMd8RBeHxrnjEFbJgTwBfTfWMrID91rYqsowlssTEa5WmDcAIOekNMmmGUCRQ9FrRbgkYcBUPV5B+4ImYNyCAqSs3d9c0mA4Nn6IqEQ5LSPYaV/JevA+KypnHeFCVCaTb3oqj//Ig6hc8bcIXweAuQDzfOj0Q5LJz4QyrsuowiPUOPrw4PNJccXLBK3LVI5XkNw20ybbmHLzQOZkuJaeIOREF/bE0xn0S9nj1FP6vi5tXwZlaZpAsT6LZUYvuZzEJqqAfWvnRWBhqVesEYDo1I4AUcEHjQ9Pl/OtbB9zrmkEMW/Olro542cxhLHTTy/my2NlsSNSwQN8zi9lGYc+citeloP3Fu161sQdtowlnLjH8hIZxuMzpKfD7vf9x7KYBLUvA2qTw+SGnidznkoF/HyCowumOZX+d1xD6fFYf3rATfDI1ugtJytIDR8bcdEpkIWFbHrWi8WNHC2GfC6poxvYjiVzQsvCybF7iCZ8m7wbV5gtYgkTBdwcsyzZbGvcH3hmRiN3qWgm5pQyDoqTyO1Hzew+MadsukzpXAtgzOjPsSBkSdtmwRWIh4Z3ckB0xF3ohbZ4KItmqYpxrWQhZhRk+StLxXyErj0uRaLDWEbWPAQ4VuqgvwyQCsjAigdPKtNQa3hM9GACXfu9EyTSOCL+6+YWJNfNtveQjLK+8tfIkQ+p7PZ3XkpCxFN2OnVctBlDFJ5KiSXgAKcJRvhaxkmh8PcjtLJAPDDb0H/+V9EsjwIiv0XM6PN59fZYn3iqELE/0ufN7ecjPrKxnq1FnKW8cI4Syxs0RuHe9PeASX/Vs/SQ0FSel7hQ8KzTspEHDRVjQNGTEOsSVmZYyLbLzMo9/0CCplGDdhtQdisrW8M7HE97nO+sw2HohbM5sDyVxd7zUi/rCFcM/6IxuK1j9JgCzySW3ObBMhRElN4CV0Iac5x1ENXGvTRteIU4Dnbou3knP3eBf/ZhoEHvTLB4RZuU+ng1n69BxqGewdq8x8GGO2zlbD9HB8G6u6r4RxH0xMraToD/fw7Oa+vreYcZQ4d5bILhvfSkk/MsSfenJxXW8vzD1+S/UEsBAhQAFAAAAAgAIXphXdrQAvRsGgAAgEwAABIAAAAAAAAAAAAAALaBAAAAAGNyb3Nzd29yZF9iZW5jaC5weVBLAQIUABQAAAAIACF6YV3xNv17fgcAAA8XAAAXAAAAAAAAAAAAAAC2gZwaAAB0ZXN0X2Nyb3Nzd29yZF9iZW5jaC5weVBLAQIUABQAAAAIAOF5YV1Qyf5LiAkAAGATAAAJAAAAAAAAAAAAAAC2gU8iAABSRUFETUUubWRQSwUGAAAAAAMAAwC8AAAA/isAAAAA")
assert hashlib.sha256(payload).hexdigest()=="0523b5fb363cfbd19672976f026f6e90cd0550535198d3d4ab67d0be829283b7"
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for name in archive.namelist():
        target=(PROJECT/name).resolve()
        if PROJECT.resolve() not in target.parents:raise ValueError('Unsafe archive path')
        content=archive.read(name)
        if target.exists() and target.read_bytes()!=content:
            raise RuntimeError('Existing source differs. Use a fresh PROJECT path.')
        if not target.exists():target.write_bytes(content)
os.chdir(PROJECT)
if str(PROJECT) not in sys.path:sys.path.insert(0,str(PROJECT))
print('Project:',PROJECT)


## 1. Check code and prepare the 50-puzzle dataset
The tests below make no network/model calls. Dataset preparation downloads the original source file and license; it does not run inference. Selection seed is separate from the LLM trial seed. Keep selected puzzles and heuristic settings fixed throughout evaluation.


In [ ]:
subprocess.run([sys.executable,'-m','unittest','test_crossword_bench','-q'],check=True)
import crossword_bench as bench
SELECTION_SEED=2026
puzzles=bench.prepare_dataset(PROJECT/'data',selection_seed=SELECTION_SEED)
assert len(puzzles)==50
print('Selected:',len(puzzles),'puzzles')
print('First five IDs:',[p['id'] for p in puzzles[:5]])
print('Source metadata:',PROJECT/'data/source_manifest.json')
# Do not inspect solutions to tune the policy on these evaluation puzzles.


## 2. Load Qwen on one GPU
Uses `Qwen/Qwen2.5-7B-Instruct` in bitsandbytes 4-bit NF4. A model revision is pinned on first load. No paid API, Ollama service or uploaded model is needed. JSON generation is prompt-based, without a constrained grammar; parsing failures are counted.

The same backend is used for both policies. Results from NF4 should not be pooled with older Ollama/GGUF experiments as if the backend were identical. Model loading can take several minutes.


In [ ]:
import torch, platform, importlib.metadata
from huggingface_hub import HfApi
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
assert torch.cuda.is_available(),'Enable GPU in Kaggle notebook settings.'
MODEL_ID='Qwen/Qwen2.5-7B-Instruct'
pin_path=PROJECT/'model_pin.json'
if pin_path.exists():
    pin=json.loads(pin_path.read_text())
    assert pin['model']==MODEL_ID
else:
    pin=dict(model=MODEL_ID,revision=HfApi().model_info(MODEL_ID).sha)
    bench.write_json(pin_path,pin)
tokenizer=AutoTokenizer.from_pretrained(MODEL_ID,revision=pin['revision'],trust_remote_code=False)
if tokenizer.pad_token_id is None:tokenizer.pad_token=tokenizer.eos_token
quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,bnb_4bit_compute_dtype=torch.float16)
model=AutoModelForCausalLM.from_pretrained(MODEL_ID,revision=pin['revision'],
    quantization_config=quant,device_map={'':0},torch_dtype=torch.float16,
    attn_implementation='sdpa',trust_remote_code=False).eval()
client=bench.HFClient(model,tokenizer,torch)
runtime=dict(**pin,backend='transformers-nf4-json-prompting',gpu=torch.cuda.get_device_name(0),
    cuda=torch.version.cuda,python=platform.python_version(),
    packages={name:importlib.metadata.version(name) for name in ('torch','transformers','accelerate','bitsandbytes','huggingface_hub')})
bench.write_json(PROJECT/'runtime.json',runtime)
(PROJECT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print('Loaded:',MODEL_ID, runtime['gpu'],pin['revision'])


## 3. Frozen comparison settings
`H = 0.35 Q + 0.30 P - 0.05 D - 0.15 C + 0.15 V`

- **Q**: judged plausibility of complete clue answers, including implied crossing words.
- **P**: judged feasibility of the full puzzle; uncalibrated, not measured success probability.
- **D**: action depth / maximum depth, multiplied by `(1-P)`.
- **C**: future-work proxy based on unassigned-slot fraction and observed mean call tokens, normalized by budget.
- **V**: minimum word-assignment distance to another retained state, gated by `sqrt(Q*P)`.

Both policies propose at most 3 moves and batch-evaluate their children. Revising a word removes conflicting crossing assignments. DFS orders siblings by proposed confidence; best-first ranks the global frontier by H. Both prune at the same P threshold and choose their final output by the same `coverage * Q` ranking. A filled grid is never assumed correct.

Default per-method limits: **30,000 tokens, 40 calls, 20 expansions**. All generation and judging costs count. The full 50-puzzle comparison can take hours. Do not tune on these 50 test answers. See the embedded README for feature definitions and limitations.


In [ ]:
import copy
config=copy.deepcopy(bench.DEFAULT)
bench.write_json(PROJECT/'config.json',config)
print(json.dumps(config,indent=2))


## 4. Run exactly 50 puzzles
Set `RUN_BENCHMARK=True` and execute this cell. Start with trial seed 42. Additional seeds repeat the same 50 puzzles; they do not select more puzzles. Both methods run for each puzzle, with alternating order.

Completed methods and successful calls can be resumed with identical settings. A failed/interrupted inference whose token cost is unknown blocks that trial; start a fresh output trial rather than hide that cost. After a hard session reset, a stale `run.lock` must only be removed when no run is active. Save the export ZIP before ending your session.


In [ ]:
RUN_BENCHMARK=False
TRIAL_SEEDS=[42]  # Optional later repetition: [42,43,44]
RUN_LABEL='main'  # Use a new label for changed settings or a fresh trial.
if RUN_BENCHMARK:
    for seed in TRIAL_SEEDS:
        output=PROJECT/'results'/f'{RUN_LABEL}_seed{seed}'
        summary=bench.run(puzzles,client,config,runtime,output,seed=seed)
        print(json.dumps(summary,indent=2))
else:
    print('Ready. Set RUN_BENCHMARK=True to start the benchmark.')


## 5. Compare scores and costs
Word accuracy is the primary score. Also report full-grid success, letter accuracy, token cost, calls and runtime. Summaries use only paired completed puzzles. The paired bootstrap resamples puzzles (not individual letters); it does not correct for related clues across puzzles or replace a larger evaluation. A better score alone does not establish SOTA.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
for directory in sorted((PROJECT/'results').glob('*')):
    if not (directory/'records').exists():continue
    summary=bench.report(directory)
    print(directory.name)
    table=pd.DataFrame(summary);display(table)
    paired_path=directory/'per_question.csv'
    if not paired_path.exists():continue
    paired=pd.read_csv(paired_path)
    delta=(paired['heuristic_bestfirst_word_accuracy']-paired['tot_dfs_word_accuracy']).to_numpy()
    rng=np.random.default_rng(123)
    means=rng.choice(delta,size=(5000,len(delta)),replace=True).mean(axis=1)
    comparison=dict(paired_puzzles=len(delta),mean_word_accuracy_difference=float(delta.mean()),
        bootstrap_95_percent_interval=np.quantile(means,[.025,.975]).tolist())
    bench.write_json(directory/'paired_comparison.json',comparison)
    print('Heuristic minus DFS:',comparison)
    fig,axes=plt.subplots(1,2,figsize=(10,4))
    axes[0].bar(table.method,table.mean_word_accuracy);axes[0].set_ylim(0,1)
    axes[0].set_title('Word accuracy');axes[0].tick_params(axis='x',rotation=15)
    axes[1].bar(table.method,table.mean_total_tokens)
    axes[1].set_title('Mean input + output tokens');axes[1].tick_params(axis='x',rotation=15)
    fig.tight_layout();fig.savefig(directory/'comparison.png',dpi=160);plt.show()


## 6. Download results or a checkpoint
Run after completing or stopping the benchmark. The ZIP includes code, exact configuration, dataset provenance, selected questions and answer keys, per-problem results, summaries, raw calls, traces and plots. It excludes model weights. Also save Kaggle notebook outputs.

For another session, attach your own ZIP as input and extract it under `/kaggle/working` before running setup; it restores the `crosswords50` directory. Resume requires the same experiment/runtime. A GPU/session interruption before a call is saved leaves unknown cost; use a fresh trial for clean cost measurement.


In [ ]:
from IPython.display import FileLink
export=Path('/kaggle/working/crosswords50_results.zip')
with zipfile.ZipFile(export,'w',zipfile.ZIP_DEFLATED) as archive:
    for path in PROJECT.rglob('*'):
        if path.is_file() and '__pycache__' not in path.parts and path.name!='run.lock':
            archive.write(path,Path('crosswords50')/path.relative_to(PROJECT))
display(FileLink(str(export)))
